## Clasificación de Textos

<img src="figs/fig-diagrama-clasificador.png" width="900">

# Entrenamiento del clasificador

# 1. Cargar el conjunto de datos

In [ ]:
# Normalización del texto

import unicodedata
import re
PUNCTUACTION = ";:,.\\-\"'/"
SYMBOLS = "()[]¿?¡!{}~<>|"
NUMBERS= "0123456789"
SKIP_SYMBOLS = set(PUNCTUACTION + SYMBOLS)
SKIP_SYMBOLS_AND_SPACES = set(PUNCTUACTION + SYMBOLS + '\t\n\r ')

def normaliza_texto(input_str,
                    punct=False,
                    accents=False,
                    num=False,
                    max_dup=2):
    """
        punct=False (elimina la puntuación, True deja intacta la puntuación)
        accents=False (elimina los acentos, True deja intactos los acentos)
        num= False (elimina los números, True deja intactos los acentos)
        max_dup=2 (número máximo de símbolos duplicados de forma consecutiva, rrrrr => rr)
    """
    
    nfkd_f = unicodedata.normalize('NFKD', input_str)
    n_str = []
    c_prev = ''
    cc_prev = 0
    for c in nfkd_f:
        if not num:
            if c in NUMBERS:
                continue
        if not punct:
            if c in SKIP_SYMBOLS:
                continue
        if not accents and unicodedata.combining(c):
            continue
        if c_prev == c:
            cc_prev += 1
            if cc_prev >= max_dup:
                continue
        else:
            cc_prev = 0
        n_str.append(c)
        c_prev = c
    texto = unicodedata.normalize('NFKD', "".join(n_str))
    texto = re.sub(r'(\s)+', r' ', texto.strip(), flags=re.IGNORECASE)
    return texto



In [ ]:
import pandas as pd
dataset = pd.read_json("./data/data_emotions_es.json", lines=True)
X = dataset['text'].to_numpy()
Y = dataset['klass'].to_numpy()

dataset.groupby(['klass']).count()

## 2. Preparar los conjuntos de datos  (datasets) para entrenamiento y para probar el rendimiento del clasificador

In [ ]:
# Dividir el conjunto de datos en conjunto de entrenamiento (80%) y conjunto de pruebas (20%)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
# Normalizar las etiquetas a una codificación ordinal para entrada del clasificador
Y_encoded= le.fit_transform(Y)


X_train, X_test, Y_train, Y_test =  train_test_split(X, Y_encoded, test_size=0.2, stratify=Y_encoded, random_state=42)


## 3. Creación de la matriz de pesos 

In [ ]:
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import word_tokenize

_STOPWORDS = stopwords.words("spanish")
# Preprocesamiento personalizado 
def mi_preprocesamiento(texto):
    #convierte a minúsculas el texto antes de normalizar
    tokens = word_tokenize(texto.lower())
    texto = " ".join(tokens)
    texto = normaliza_texto(texto)
    return texto
    
# Tokenizador personalizado 
def mi_tokenizador(texto):
    # Elimina stopwords 
    return [t for t in texto.split() if t not in _STOPWORDS]


vec_tfidf = TfidfVectorizer(analyzer="word", preprocessor=mi_preprocesamiento, tokenizer=mi_tokenizador,  ngram_range=(1,1))
X_tfidf = vec_tfidf.fit_transform(X_train)
print("tamaño de la matriz: ", X_tfidf.shape)
print("vocabulario: ", len(vec_tfidf.get_feature_names_out()))


## 4. Convierte los ejemplos de prueba al mismo espacio de representación de datos del conjunto de entrenamiento que aprendió el modelo

In [ ]:

X_test_tfidf = vec_tfidf.transform(X_test)


# 5. Entrenar al clasificador

### Clasificador: Support Vector Machine (SVM)

- ### El **Método de Máquinas de Vectores de Soporte (SVM, por sus siglas en inglés)** 
    - #### Es un algoritmo de aprendizaje supervisado utilizado para clasificación y regresión. 
    - #### Su objetivo principal es encontrar un hiperplano en un espacio de características que maximice el margen entre diferentes clases de datos.

- ### Funcionamiento:

    1. **Clasificación**: 
        - SVM busca el hiperplano que mejor separa las diferentes clases de datos con el mayor margen posible. 
        - Los puntos de datos que están más cerca del hiperplano se llaman vectores de soporte, ya que definen el margen de separación.

    2. **Regresión**: 
        - En el caso de regresión, SVM intenta encontrar un hiperplano que se ajuste a los datos de manera que la desviación entre las predicciones y los valores reales sea mínima.

- ### Características Clave:

    - **Hiperplano Óptimo**: Maximiza la distancia entre el hiperplano y los vectores de soporte. En la fig. H3 maximiza el margen
    - **Espacio de Características**: Puede utilizar el truco del núcleo (kernel trick) para transformar los datos en un espacio de dimensiones más altas, facilitando la separación no lineal.
    - **Regularización**: Controla el equilibrio entre el margen máximo y la clasificación correcta de los datos de entrenamiento.



<center>
<img src="figs/fig-svm.png" width="300" style="background-color:white;">
</center>

###### Fuente: https://es.wikipedia.org/wiki/Máquinas_de_vectores_de_soporte


### Importamos SVC del paquete sklearn

In [ ]:
from sklearn.svm import SVC

clasificador_svc = SVC(kernel="linear")

# Entrenamos al clasificador

clasificador_svc.fit(X_tfidf, Y_train)




## Ajuste de hiperparámetros: `C` y `gamma`

---

### `C` — Regularización (cuánto castigas el error)

`C` controla cuánto le importa al modelo equivocarse durante el entrenamiento.

| `C` | Comportamiento | Síntoma |
|-----|----------------|---------|
| **Bajo** (0.01, 0.1) | Tolera errores. Margen ancho. | Underfitting: train y test bajos |
| **Medio** (1, 10) | Equilibrado | Buen balance |
| **Alto** (100, 1000) | Castiga cada error. Margen estrecho. | Overfitting: train alto, test bajo |

**Intuición:** `C` es "qué tan enojado te pones cuando el modelo se equivoca".

- `C` alto = perfeccionista → memoriza
- `C` bajo = flexible → generaliza pero puede ignorar casos

**Qué probar:**

```python
C_grid = [0.01, 0.1, 1, 10, 100, 1000]
```

Empieza en `[0.1, 1, 10]`, amplía si hace falta.

---

### `gamma` — Alcance de la influencia

Solo aplica a kernels `rbf`, `poly` y `sigmoid`. Controla qué tan lejos 
llega la influencia de cada ejemplo de entrenamiento.

| `gamma` | Comportamiento | Síntoma |
|---------|----------------|---------|
| **Bajo** (0.001, 0.01) | Influencia lejana. Frontera suave. | Underfitting |
| **Medio** (0.1, 1) | Equilibrado | Buen balance |
| **Alto** (10, 100) | Influencia muy local. Frontera enredada. | Overfitting |

**Intuición:** `gamma` es "el zoom con el que miras los datos".

- `gamma` alto = lupa → ve detalles y ruido
- `gamma` bajo = vista panorámica → ve solo lo general

**Valores típicos:**

- `'scale'` (default): `1 / (n_features * X.var())`
- `'auto'`: `1 / n_features`
- Manual: `[0.001, 0.01, 0.1, 1, 10]`

> **Nota con TF-IDF:** como hay miles de features, `gamma='scale'` suele 
> quedar muy pequeño. A veces conviene subirlo manualmente.

---

### Cómo se combinan

| Train | Test | Problema | Solución |
|-------|------|----------|----------|
| Alto  | Bajo | Overfitting | Bajar `C` y/o `gamma` |
| Bajo  | Bajo | Underfitting | Subir `C` y/o `gamma` |
| Alto  | Alto |  Bien | No mover |

---

### Predicción de los datos del conjuntos de prueba con el modelo entrenado

In [ ]:
y_pred = clasificador_svc.predict(X_test_tfidf)


### Inspección de los resultados de los primeros N ejemplos de prueba

In [ ]:
print("textos: ", X_test[:5])
print("clase esperada: ",Y_test[:5])
print("clase predicha: ", y_pred[:5])

### Mostrar la predicción de la clase original 

In [ ]:
# Obten las primeras N predicciones
pred =  y_pred[:5] 
pred_ori = le.inverse_transform(pred)
pred, pred_ori

# 6. Evaluando el desempeño

## Métricas de Evaluación
 - #### Las métricas precisión, recall y F1 son fundamentales para evaluar el rendimiento de un clasificador


<img src="figs/fig_precision-recall.png" width="300">

##### Fuente: https://en.wikipedia.org/wiki/Precision_and_recall


<img src="figs/fig_matriz-confusion.png" width="500">


TP=True Positive

TN=True Negative

FP=False Positive (Error tipo I: ejemplo, se considera que el paciente está enfermo, pero en realidad está sano)

FN=False Negative ( Error tipo II: ejemplo, se considera que el paciente está sano, pero en realidad está enfermo)


$$ Accuracy = \frac{total~ TP + total~TN}{total~muestras} $$

$$ Precision_c = \frac{ TP_c}{TP_c + FP_c} $$

$$ Recall_c = \frac{ TP_c}{TP_c + FN_c} $$

$$ F1-score_c= 2 \times \frac{ Precision_c \times Recall_c}{Precision_c + Recall_c} $$

$$ macro-F1-score= \frac{ 1 }{|Clases|} \sum{F1-score_c} $$

## Matriz de confusión

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(Y_test, y_pred)

In [ ]:
# para la clase 0, la precisión es la siguiente
tp= 119
fp = 12+10+10
tp/(tp + fp)

## Métricas

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
print("P=", precision_score(Y_test, y_pred, average='macro'))
print("R=", recall_score(Y_test, y_pred, average='macro'))
print("F1=", f1_score(Y_test, y_pred, average='macro'))
print("Acc=", accuracy_score(Y_test, y_pred))


## Inspección del desempeño por clase

In [ ]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(Y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(Y_test, y_pred, digits=4, zero_division='warn'))

## 7. Probar el modelo con nuevos datos

### Una vez creado el modelo se puede probar el modelo con datos que no se entrenó

In [ ]:
textos = [
    "Hoy es un día maravilloso, estoy muy feliz",
    "No soporto esta situación, me da mucha rabia",
]
# 1. Transformar los datos al espacio de representación

X_nuevo = vec_tfidf.transform(textos)

# 2. Predecir
y_pred_enc = clasificador_svc.predict(X_nuevo)

# 3. Decodificar a etiquetas originales
y_pred = le.inverse_transform(y_pred_enc)

# 4. Imprimir los resultados 

for texto, emocion in zip(textos, y_pred):
    print(f"[{emocion:10}] {texto}")

# Ejercicio: SVM con kernel RBF

## Objetivo

Hasta ahora usamos `SVC(kernel="linear")`. En este ejercicio probar 
el **kernel RBF** (Radial Basis Function), que permite fronteras **no lineales**.

## Instrucciones

1. **Entrenar un SVC con kernel RBF** usando los mismos datos vectorizados 
   (`X_tfidf`, `Y_train`) que ya tienen.
2. **Predecir sobre test** (`X_test_tfidf`) y calcular:
   - `accuracy_score`
   - `f1_score` con `average="macro"`
3. **Comparar** los resultados con el modelo lineal que ya entrenaron.
4. **Probar 3 valores de `gamma`** (`0.001`, `0.01`, `0.1`) manteniendo `C=1.0`.
5. **Responder** al final: ¿mejoró el kernel RBF? 
## Pistas

- `SVC(kernel="rbf", C=1.0, gamma=0.01)`
- Para medir: `from sklearn.metrics import accuracy_score, f1_score`
- `f1_score(y_true, y_pred, average="macro")` da el promedio por clase.
- La celda de `TfidfVectorizer` y `train_test_split` **no se toca**. Se usan los mismos preprocesamientos y misma partición de datos.